# Advanced Tutorial Problems with Solutions: Decorator Classes

This notebook develops **advanced problems with complete solutions** for class-based decorators.

The style is intentionally tutorial-like: instead of jumping straight to a final implementation, each problem is broken into logical steps. We will repeatedly:

1. identify the behavior we want,
2. predict what Python will do,
3. build a minimal example,
4. inspect the result,
5. improve the design,
6. verify the solution with assertions.

The main theme is that a class-based decorator used on methods is not merely a callable wrapper—it can also need to participate in Python's **descriptor protocol**.

## What this notebook covers

We will work through advanced examples involving:

- callable decorator objects with `__call__`,
- why method binding breaks for naive decorator classes,
- `__get__` and `types.MethodType`,
- metadata preservation with `update_wrapper`,
- parameterized decorator classes,
- shared versus per-instance state,
- `WeakKeyDictionary`,
- descriptor lookup behavior,
- data versus non-data descriptors,
- `__set_name__`,
- decorator order with `classmethod` and `staticmethod`,
- signature-aware validation,
- retry behavior,
- timing and exception statistics,
- thread-safe decorator state,
- per-instance memoization,
- inheritance-aware descriptors,
- a larger capstone instrumentation decorator.

## Imports used throughout the notebook

Everything here uses only the Python standard library.

In [1]:
from functools import update_wrapper
from types import MethodType
from weakref import WeakKeyDictionary
from collections import defaultdict
from time import perf_counter, sleep
import inspect
import threading

# Problem 1 — Why a callable decorator object breaks method binding

We begin with the simplest class-based decorator possible.

The decorator works on ordinary functions because the object returned by the decorator only needs to be callable.

The interesting question is what happens when the same object is stored as an attribute on a class and is expected to behave like a method.

## Step 1 — Build the minimal callable decorator

The constructor receives the function being decorated.

The decorator object stores that function.

The `__call__` method forwards arguments to the original function.

In [2]:
class SimpleLogger:
    def __init__(self, fn):
        self.fn = fn

    def __call__(self, *args, **kwargs):
        print(f"calling {self.fn.__name__}")
        return self.fn(*args, **kwargs)


@SimpleLogger
def add(a, b):
    return a + b


assert add(2, 3) == 5
print("decorated object type:", type(add).__name__)

calling add
decorated object type: SimpleLogger


The name `add` no longer refers to a function. It refers to a `SimpleLogger` instance.

That is not a problem for a free function: calling the object invokes its `__call__` method, which then calls the stored function.

## Step 2 — Put the same decorator on an instance method

Before running the next cell, predict the result of `box.label()`.

A normal Python method call automatically supplies the instance as the first argument. But that behavior comes from functions acting as descriptors.

Our decorator object is callable, but it is not yet a descriptor.

In [3]:
class Box:
    def __init__(self, value):
        self.value = value

    @SimpleLogger
    def label(self):
        return f"Box({self.value})"


box = Box(10)

try:
    box.label()
except TypeError as ex:
    print("Expected failure:")
    print(ex)

calling label
Expected failure:
Box.label() missing 1 required positional argument: 'self'


## Explanation

The decorated attribute stored on `Box` is a `SimpleLogger` object.

Python does not automatically bind arbitrary callable objects to instances.

The original function knew how to become a bound method because functions implement the descriptor protocol. Our wrapper object does not yet implement the relevant hook: `__get__`.

# Problem 2 — Make the decorator behave like a method descriptor

We now want the decorator object to support two forms of attribute access:

- `Class.method` — access through the class,
- `instance.method` — access through an instance.

The method that controls this is `__get__`.

## Step 1 — Handle class access

When Python calls:

```python
wrapper.__get__(None, OwnerClass)
```

there is no instance to bind.

A useful default is to return the wrapper object itself so that class-level inspection still works.

## Step 2 — Handle instance access

When Python supplies a real instance, we need a callable that inserts that instance as the first positional argument.

`MethodType(callable, instance)` creates exactly that kind of bound method.

In [4]:
class BindingLogger:
    def __init__(self, fn):
        self.fn = fn

    def __call__(self, *args, **kwargs):
        print(f"calling {self.fn.__name__}")
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

## Step 3 — Verify the two access paths

We will check that:

- class access gives us the descriptor object,
- instance access gives a bound method,
- calling the bound method supplies the instance correctly.

In [5]:
class Product:
    def __init__(self, name):
        self.name = name

    @BindingLogger
    def describe(self):
        return f"Product: {self.name}"


p = Product("Keyboard")

assert isinstance(Product.describe, BindingLogger)
assert inspect.ismethod(p.describe)
assert p.describe() == "Product: Keyboard"

print("class access:", Product.describe)
print("instance access:", p.describe)

calling describe
class access: <__main__.BindingLogger object at 0x0000022AE77B2900>
instance access: <bound method ? of <__main__.Product object at 0x0000022AE77B2A50>>


## Reusable pattern

For many class-based decorators used on instance methods, this is the essential binding pattern:

```python
def __get__(self, instance, owner):
    if instance is None:
        return self
    return MethodType(self, instance)
```

This turns the callable wrapper object into a non-data descriptor with method-like binding behavior.

# Problem 3 — Preserve metadata on a decorator instance

Binding is now correct, but introspection is still weak.

A good decorator should preserve information such as the original callable's:

- name,
- qualified name,
- docstring,
- annotations,
- wrapped-object reference.

## Step 1 — Observe the metadata gap

A custom wrapper object does not automatically inherit the metadata of the function it stores.

In [6]:
def sample(x: int) -> int:
    "Return x doubled."
    return x * 2


raw = BindingLogger(sample)

print("function name:", sample.__name__)
print("wrapper class:", type(raw).__name__)
print("wrapper has __wrapped__:", hasattr(raw, "__wrapped__"))

function name: sample
wrapper class: BindingLogger
wrapper has __wrapped__: False


## Step 2 — Use `update_wrapper`

`functools.wraps` is most convenient when the wrapper itself is a function.

For a wrapper **object**, `functools.update_wrapper` is a direct fit.

We call it in the decorator object's constructor.

In [7]:
class MetadataLogger:
    def __init__(self, fn):
        self.fn = fn
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

## Step 3 — Verify metadata and unwrapping

The `inspect` module understands the `__wrapped__` convention created by `update_wrapper`.

In [8]:
class Calculator:
    @MetadataLogger
    def square(self, x: int) -> int:
        "Return x squared."
        return x * x


assert Calculator.square.__name__ == "square"
assert Calculator.square.__doc__ == "Return x squared."
assert inspect.unwrap(Calculator.square).__name__ == "square"
assert Calculator().square(7) == 49

print(Calculator.square.__name__)
print(Calculator.square.__annotations__)
print(inspect.signature(Calculator.square))

square
{'x': <class 'int'>, 'return': <class 'int'>}
(self, x: int) -> int


# Problem 4 — Build a parameterized class-based decorator cleanly

So far the decorator constructor directly receives the function:

```python
@MetadataLogger
def f(...):
    ...
```

Now we want syntax such as:

```python
@Tagged("database")
def f(...):
    ...
```

That creates an extra stage: configuration happens before decoration.

## Step 1 — Identify the lifecycle

With `@Tagged("database")`, Python first evaluates `Tagged("database")`.

That produces an object.

Python then calls that object with the function being decorated.

This means one object can represent configuration and another object can represent the callable descriptor.

## Step 2 — Separate configuration from wrapping

Using two cooperating classes often keeps the lifecycle clearer:

- `Tagged` stores decorator options,
- `_TaggedWrapper` wraps one specific function and implements descriptor behavior.

In [9]:
class Tagged:
    def __init__(self, tag):
        self.tag = tag

    def __call__(self, fn):
        return _TaggedWrapper(fn, self.tag)


class _TaggedWrapper:
    def __init__(self, fn, tag):
        self.fn = fn
        self.tag = tag
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        print(f"[{self.tag}] {self.fn.__name__}")
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

## Step 3 — Verify free functions and methods

The same wrapper can support both because `__get__` only participates when the object is retrieved as a class attribute.

In [10]:
@Tagged("math")
def multiply(a, b):
    return a * b


class Repository:
    @Tagged("storage")
    def save(self, item):
        return f"saved {item}"


assert multiply(4, 5) == 20
assert Repository().save("record") == "saved record"
assert Repository.save.tag == "storage"

[math] multiply
[storage] save


# Problem 5 — Shared state versus per-instance state

One of the best reasons to use a class-based decorator is persistent state.

But there is an important design question:

> Is the state shared across all instances, or should every object have independent state?

## Step 1 — Store the count on the descriptor itself

The decorator object is created when the class body executes and is stored on the class.

Therefore, a counter stored directly on the wrapper is shared by every instance using that method.

In [11]:
class SharedCallCounter:
    def __init__(self, fn):
        self.fn = fn
        self.calls = 0
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)


class Worker:
    def __init__(self, name):
        self.name = name

    @SharedCallCounter
    def work(self):
        return self.name

## Step 2 — Observe shared accumulation

Calls through different `Worker` instances all update the same descriptor object.

In [12]:
w1 = Worker("A")
w2 = Worker("B")

w1.work()
w1.work()
w2.work()

assert Worker.work.calls == 3
print("shared call count:", Worker.work.calls)

shared call count: 3


## Design checkpoint

Shared descriptor state is correct for questions like:

> How many total calls has this method received?

It is not correct for questions like:

> How many times has each individual object called this method?

That requires per-instance storage.

# Problem 6 — Keep per-instance state with `WeakKeyDictionary`

We want the decorator to keep a separate count for each instance without writing bookkeeping attributes directly onto the user's objects.

A `WeakKeyDictionary` is a natural fit because it can associate metadata with instances without necessarily keeping those instances alive forever.

## Step 1 — Store counts externally

The wrapper owns a weak mapping:

```python
instance -> call_count
```

The decorated instance becomes the first argument when `MethodType` binds the wrapper.

In [13]:
class PerInstanceCounter:
    def __init__(self, fn):
        self.fn = fn
        self._counts = WeakKeyDictionary()
        update_wrapper(self, fn)

    def __call__(self, instance, *args, **kwargs):
        self._counts[instance] = self._counts.get(instance, 0) + 1
        return self.fn(instance, *args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def count_for(self, instance):
        return self._counts.get(instance, 0)

## Step 2 — Verify independent counts

We retrieve the descriptor through the class to query its state API.

In [14]:
class Sensor:
    @PerInstanceCounter
    def read(self, value):
        return value


s1 = Sensor()
s2 = Sensor()

s1.read(10)
s1.read(20)
s2.read(30)

counter = Sensor.read

assert counter.count_for(s1) == 2
assert counter.count_for(s2) == 1

print("s1:", counter.count_for(s1))
print("s2:", counter.count_for(s2))

s1: 2
s2: 1


## Practical note

Weak-reference based storage requires keys that support weak references.

Most normal user-defined class instances do, but some specialized objects do not. A reusable decorator should document that requirement if it depends on weak references.

# Problem 7 — Distinguish attribute lookup from method invocation

A descriptor's `__get__` method runs during attribute lookup.

That means simply evaluating:

```python
obj.method
```

can already execute descriptor logic, even if the result is never called.

## Step 1 — Trace class access, instance access, and actual calls separately

These are three distinct events and should not be conflated in instrumentation.

In [15]:
class AccessTracer:
    def __init__(self, fn):
        self.fn = fn
        self.class_accesses = 0
        self.instance_accesses = 0
        self.calls = 0
        update_wrapper(self, fn)

    def __get__(self, instance, owner):
        if instance is None:
            self.class_accesses += 1
            return self

        self.instance_accesses += 1
        return MethodType(self, instance)

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.fn(*args, **kwargs)

## Step 2 — Perform lookups without always calling

We use the class dictionary afterward so that we can inspect the raw descriptor without triggering another `__get__` call.

In [16]:
class Demo:
    @AccessTracer
    def ping(self):
        return "pong"


d = Demo()

_ = Demo.ping
_ = d.ping
result = d.ping()

tracer = Demo.__dict__["ping"]

assert result == "pong"
assert tracer.class_accesses == 1
assert tracer.instance_accesses == 2
assert tracer.calls == 1

print("class accesses:", tracer.class_accesses)
print("instance accesses:", tracer.instance_accesses)
print("calls:", tracer.calls)

class accesses: 1
instance accesses: 2
calls: 1


## Debugging technique

`Class.__dict__[name]` is useful when studying descriptors because it retrieves the object stored directly in the class namespace rather than performing ordinary descriptor-driven attribute access.

# Problem 8 — Non-data versus data descriptor behavior

A descriptor that implements `__get__` but not `__set__` or `__delete__` is generally a **non-data descriptor**.

Normal functions are non-data descriptors too.

One important consequence is that an instance attribute can shadow the descriptor.

## Step 1 — Demonstrate instance-level shadowing

We first use a decorator that only implements `__get__`.

In [17]:
class Shadowable:
    def __init__(self, fn):
        self.fn = fn
        update_wrapper(self, fn)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def __call__(self, *args, **kwargs):
        return self.fn(*args, **kwargs)


class Greeter:
    @Shadowable
    def greet(self):
        return "descriptor version"


g = Greeter()
assert g.greet() == "descriptor version"

g.greet = lambda: "instance version"
assert g.greet() == "instance version"

print(g.greet())

instance version


## Step 2 — Turn the wrapper into a data descriptor

Adding `__set__` changes lookup precedence.

For this problem we will reject attempts to replace the decorated method on an individual instance.

In [18]:
class ProtectedDecorator(Shadowable):
    def __set__(self, instance, value):
        raise AttributeError(f"{self.__name__} cannot be replaced on an instance")


class SecureGreeter:
    @ProtectedDecorator
    def greet(self):
        return "protected descriptor version"


sg = SecureGreeter()

try:
    sg.greet = lambda: "replacement"
except AttributeError as ex:
    print(ex)

assert sg.greet() == "protected descriptor version"

greet cannot be replaced on an instance


## Key lesson

Adding `__set__` is not just adding assignment behavior.

It changes the descriptor's category and therefore changes attribute lookup precedence.

Descriptor methods are part of a protocol, so small changes can have larger semantic effects.

# Problem 9 — Discover the final attribute name with `__set_name__`

A descriptor can be told the name under which it was assigned in the class body.

Python calls:

```python
descriptor.__set_name__(owner, name)
```

during class creation.

## Step 1 — Capture the owner and attribute name

This can be useful for diagnostics, logging, registration, and validation.

In [19]:
class NamedLogger:
    def __init__(self, fn):
        self.fn = fn
        self.owner = None
        self.attribute_name = None
        update_wrapper(self, fn)

    def __set_name__(self, owner, name):
        self.owner = owner
        self.attribute_name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def __call__(self, *args, **kwargs):
        print(f"{self.owner.__name__}.{self.attribute_name}")
        return self.fn(*args, **kwargs)

## Step 2 — Verify class-creation metadata

In [20]:
class Service:
    @NamedLogger
    def execute(self, x):
        return x + 1


descriptor = Service.__dict__["execute"]

assert descriptor.owner is Service
assert descriptor.attribute_name == "execute"
assert Service().execute(9) == 10

Service.execute


# Problem 10 — Decorator order with `classmethod` and `staticmethod`

Stacked decorators are applied from the bottom upward.

Therefore:

```python
@A
@B
def f(...):
    ...
```

is equivalent to:

```python
f = A(B(f))
```

This matters greatly when some decorators are themselves descriptors.

## Step 1 — Let the custom decorator wrap the raw function first

A straightforward ordering is:

```python
@classmethod
@Verbose
```

or:

```python
@staticmethod
@Verbose
```

Then the built-in descriptor remains the outermost object stored in the class dictionary.

In [21]:
class Verbose:
    def __init__(self, fn):
        self.fn = fn
        update_wrapper(self, fn)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def __call__(self, *args, **kwargs):
        print(f"Verbose -> {self.fn.__name__}")
        return self.fn(*args, **kwargs)


class Utilities:
    @classmethod
    @Verbose
    def class_name(cls):
        return cls.__name__

    @staticmethod
    @Verbose
    def triple(x):
        return x * 3


assert Utilities.class_name() == "Utilities"
assert Utilities.triple(4) == 12

Verbose -> class_name
Verbose -> triple


## Step 2 — Inspect the objects actually stored in the class

This is often the fastest way to understand stacked descriptor decorators.

In [22]:
assert isinstance(Utilities.__dict__["class_name"], classmethod)
assert isinstance(Utilities.__dict__["triple"], staticmethod)

print(type(Utilities.__dict__["class_name"]).__name__)
print(type(Utilities.__dict__["triple"]).__name__)

classmethod
staticmethod


## Why the order works

`Verbose` first wraps the plain function.

Then `classmethod` or `staticmethod` wraps that callable object.

The built-in descriptor remains responsible for its normal binding semantics.

# Problem 11 — Signature-aware argument validation

A sophisticated decorator can inspect the original function signature and reason about arguments by parameter name rather than manually guessing positions in `args`.

## Step 1 — Compute the signature once

The decorated function is fixed after decoration, so there is no need to call `inspect.signature` on every invocation.

We store the signature in the wrapper constructor.

In [23]:
class PositiveArguments:
    def __init__(self, fn):
        self.fn = fn
        self.signature = inspect.signature(fn)
        update_wrapper(self, fn)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def __call__(self, *args, **kwargs):
        bound = self.signature.bind(*args, **kwargs)
        bound.apply_defaults()

        for name, value in bound.arguments.items():
            if name in {"self", "cls"}:
                continue
            if isinstance(value, (int, float)) and value <= 0:
                raise ValueError(f"{name} must be positive")

        return self.fn(*args, **kwargs)

## Step 2 — Verify positional, keyword, and default handling

In [24]:
class Geometry:
    @PositiveArguments
    def rectangle_area(self, width, height=1):
        return width * height


geo = Geometry()

assert geo.rectangle_area(3, 4) == 12
assert geo.rectangle_area(width=5) == 5

try:
    geo.rectangle_area(-2, 4)
except ValueError as ex:
    print(ex)

width must be positive


## Why `Signature.bind` is useful

It gives us Python's own argument-binding rules instead of reimplementing them.

That makes the decorator easier to reason about when functions contain positional, keyword, defaulted, or keyword-only parameters.

# Problem 12 — A configurable retry decorator with persistent statistics

A class-based decorator is a natural fit when we need both configuration and state.

We will build a retry decorator that remembers how many caught failures have occurred over its lifetime.

## Step 1 — Separate configuration from the runtime wrapper

`Retry(...)` stores the options.

Calling that configuration object with a function creates the actual descriptor wrapper.

In [25]:
class Retry:
    def __init__(self, attempts=3, exceptions=(Exception,)):
        if attempts < 1:
            raise ValueError("attempts must be >= 1")
        self.attempts = attempts
        self.exceptions = exceptions

    def __call__(self, fn):
        return _RetryWrapper(fn, self.attempts, self.exceptions)


class _RetryWrapper:
    def __init__(self, fn, attempts, exceptions):
        self.fn = fn
        self.attempts = attempts
        self.exceptions = exceptions
        self.failures = 0
        update_wrapper(self, fn)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def __call__(self, *args, **kwargs):
        last_error = None

        for _ in range(self.attempts):
            try:
                return self.fn(*args, **kwargs)
            except self.exceptions as ex:
                self.failures += 1
                last_error = ex

        raise last_error

## Step 2 — Test transient failures followed by success

The service instance owns one kind of state (`internal_attempts`).

The decorator object owns another kind of state (`failures`).

In [26]:
class FlakyService:
    def __init__(self):
        self.internal_attempts = 0

    @Retry(attempts=4, exceptions=(RuntimeError,))
    def fetch(self):
        self.internal_attempts += 1
        if self.internal_attempts < 3:
            raise RuntimeError("temporary failure")
        return "ok"


service = FlakyService()
assert service.fetch() == "ok"

wrapper = FlakyService.__dict__["fetch"]
assert wrapper.failures == 2
print("decorator-recorded failures:", wrapper.failures)

decorator-recorded failures: 2


# Problem 13 — Collect timing statistics, including failing calls

Printing a duration is easy.

A more useful instrumentation decorator can accumulate statistics over time.

We will track:

- number of calls,
- total elapsed time,
- minimum duration,
- maximum duration,
- average duration.

## Step 1 — Put bookkeeping in `finally`

If timing should include failing calls, the timing update must happen whether the wrapped callable returns normally or raises.

A `finally` block expresses that requirement directly.

In [27]:
class Timed:
    def __init__(self, fn):
        self.fn = fn
        self.calls = 0
        self.total_seconds = 0.0
        self.min_seconds = None
        self.max_seconds = None
        update_wrapper(self, fn)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def __call__(self, *args, **kwargs):
        start = perf_counter()
        try:
            return self.fn(*args, **kwargs)
        finally:
            elapsed = perf_counter() - start
            self.calls += 1
            self.total_seconds += elapsed
            self.min_seconds = elapsed if self.min_seconds is None else min(self.min_seconds, elapsed)
            self.max_seconds = elapsed if self.max_seconds is None else max(self.max_seconds, elapsed)

    @property
    def average_seconds(self):
        return self.total_seconds / self.calls if self.calls else 0.0

## Step 2 — Verify relationships instead of exact timings

Exact timing values vary from machine to machine, so robust tests should avoid requiring one exact duration.

In [28]:
class Job:
    @Timed
    def run(self, delay):
        sleep(delay)
        return delay


job = Job()
job.run(0.001)
job.run(0.002)
job.run(0.001)

timer = Job.__dict__["run"]

assert timer.calls == 3
assert timer.total_seconds > 0
assert timer.min_seconds <= timer.average_seconds <= timer.max_seconds

print("calls:", timer.calls)
print("average seconds:", round(timer.average_seconds, 6))

calls: 3
average seconds: 0.001619


# Problem 14 — Record exceptions without swallowing them

Instrumentation often needs to observe failures while preserving the original application behavior.

The wrapper should record the exception type and then re-raise the exception.

## Step 1 — Count failures by exception class

`defaultdict(int)` gives us a compact counter keyed by exception type.

In [29]:
class ExceptionStats:
    def __init__(self, fn):
        self.fn = fn
        self.counts = defaultdict(int)
        update_wrapper(self, fn)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def __call__(self, *args, **kwargs):
        try:
            return self.fn(*args, **kwargs)
        except Exception as ex:
            self.counts[type(ex)] += 1
            raise

## Step 2 — Verify that the caller still sees the original failures

In [30]:
class Parser:
    @ExceptionStats
    def to_int(self, text):
        return int(text)


parser = Parser()
assert parser.to_int("12") == 12

for bad in ["x", "4.2"]:
    try:
        parser.to_int(bad)
    except ValueError:
        pass

stats = Parser.__dict__["to_int"]
assert stats.counts[ValueError] == 2
print(dict(stats.counts))

{<class 'ValueError'>: 2}


# Problem 15 — Make shared decorator state thread-safe

A stateful wrapper can be invoked concurrently.

If a counter or statistics structure must be correct across threads, access to shared mutable state should be synchronized.

## Step 1 — Keep the lock next to the state it protects

The critical section should be small: we only need to protect the wrapper's counter update, not the entire wrapped function call.

In [31]:
class ThreadSafeCounter:
    def __init__(self, fn):
        self.fn = fn
        self.calls = 0
        self._lock = threading.Lock()
        update_wrapper(self, fn)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def __call__(self, *args, **kwargs):
        with self._lock:
            self.calls += 1
        return self.fn(*args, **kwargs)

## Step 2 — Run multiple calls concurrently

After joining all threads, the wrapper should report exactly the number of completed invocations.

In [32]:
class ConcurrentWorker:
    @ThreadSafeCounter
    def work(self, x):
        return x * x


worker = ConcurrentWorker()
results = []
results_lock = threading.Lock()

def task(i):
    value = worker.work(i)
    with results_lock:
        results.append(value)

threads = [threading.Thread(target=task, args=(i,)) for i in range(20)]

for t in threads:
    t.start()
for t in threads:
    t.join()

counter = ConcurrentWorker.__dict__["work"]
assert counter.calls == 20
assert len(results) == 20
print("thread-safe calls:", counter.calls)

thread-safe calls: 20


# Problem 16 — Per-instance memoization managed by the descriptor

Memoizing methods raises a state-isolation question.

Usually, different instances should not share cached results unless that sharing is intentional.

We can combine `WeakKeyDictionary` with one normal cache dictionary per instance.

## Step 1 — Build a cache key

For this tutorial, we assume positional and keyword argument values are hashable.

A production cache should document or enforce its key requirements explicitly.

In [33]:
class PerInstanceMemoize:
    def __init__(self, fn):
        self.fn = fn
        self._caches = WeakKeyDictionary()
        self.computations = 0
        update_wrapper(self, fn)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def __call__(self, instance, *args, **kwargs):
        cache = self._caches.setdefault(instance, {})
        key = (args, tuple(sorted(kwargs.items())))

        if key not in cache:
            cache[key] = self.fn(instance, *args, **kwargs)
            self.computations += 1

        return cache[key]

## Step 2 — Verify that two instances have separate caches

The same argument value is computed once for each instance.

In [34]:
class Scaler:
    def __init__(self, factor):
        self.factor = factor

    @PerInstanceMemoize
    def scale(self, value):
        return self.factor * value


a = Scaler(2)
b = Scaler(10)

assert a.scale(5) == 10
assert a.scale(5) == 10
assert b.scale(5) == 50
assert b.scale(5) == 50

memo = Scaler.__dict__["scale"]
assert memo.computations == 2
print("actual computations:", memo.computations)

actual computations: 2


# Problem 17 — Inheritance and the `owner` argument of `__get__`

A descriptor may be physically stored on a base class but accessed through a subclass.

The `owner` argument tells the descriptor which class is participating in the lookup.

## Step 1 — Record every owner observed during descriptor access

In [35]:
class OwnerAware:
    def __init__(self, fn):
        self.fn = fn
        self.owners_seen = []
        update_wrapper(self, fn)

    def __get__(self, instance, owner):
        self.owners_seen.append(owner)
        if instance is None:
            return self
        return MethodType(self, instance)

    def __call__(self, *args, **kwargs):
        return self.fn(*args, **kwargs)

## Step 2 — Access the inherited descriptor through both base and child classes

In [36]:
class Base:
    @OwnerAware
    def identify(self):
        return type(self).__name__


class Child(Base):
    pass


_ = Base.identify
_ = Child.identify
assert Child().identify() == "Child"

descriptor = Base.__dict__["identify"]

assert Base in descriptor.owners_seen
assert Child in descriptor.owners_seen
print([owner.__name__ for owner in descriptor.owners_seen])

['Base', 'Child', 'Child']


## Interpretation

The descriptor object lives in `Base.__dict__`, but `owner` can be `Child` when the attribute is resolved through the subclass.

That can be valuable for inheritance-aware registration, logging, or policy decisions.

# Problem 18 — Capstone: a reusable instrumentation decorator

We will now combine several ideas from the notebook.

Requirements:

1. support free functions and instance methods,
2. preserve metadata,
3. accept configuration,
4. learn the final attribute name with `__set_name__`,
5. count calls,
6. count exceptions by type,
7. measure total elapsed time,
8. update shared statistics safely across threads,
9. re-raise application exceptions normally,
10. expose a snapshot API.

## Step 1 — Separate configuration and the runtime wrapper

`Instrument(label=...)` is the decorator factory.

`_InstrumentedCallable` is the callable descriptor that owns runtime statistics.

## Step 2 — Decide what needs synchronization

The statistics are shared state:

- `calls`,
- `total_seconds`,
- exception counts.

We protect updates to those fields with a lock.

The wrapped function itself runs outside the lock so independent calls do not serialize unnecessarily.

In [37]:
class Instrument:
    def __init__(self, label="default"):
        self.label = label

    def __call__(self, fn):
        return _InstrumentedCallable(fn, self.label)


class _InstrumentedCallable:
    def __init__(self, fn, label):
        self.fn = fn
        self.label = label
        self.owner = None
        self.attribute_name = fn.__name__

        self.calls = 0
        self.total_seconds = 0.0
        self.exceptions = defaultdict(int)
        self._lock = threading.Lock()

        update_wrapper(self, fn)

    def __set_name__(self, owner, name):
        self.owner = owner
        self.attribute_name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def __call__(self, *args, **kwargs):
        start = perf_counter()

        try:
            return self.fn(*args, **kwargs)
        except Exception as ex:
            with self._lock:
                self.exceptions[type(ex)] += 1
            raise
        finally:
            elapsed = perf_counter() - start
            with self._lock:
                self.calls += 1
                self.total_seconds += elapsed

    @property
    def average_seconds(self):
        with self._lock:
            return self.total_seconds / self.calls if self.calls else 0.0

    def snapshot(self):
        with self._lock:
            return {
                "label": self.label,
                "owner": None if self.owner is None else self.owner.__name__,
                "attribute_name": self.attribute_name,
                "calls": self.calls,
                "total_seconds": self.total_seconds,
                "average_seconds": self.total_seconds / self.calls if self.calls else 0.0,
                "exceptions": {
                    exc_type.__name__: count
                    for exc_type, count in self.exceptions.items()
                },
            }

## Step 3 — Use the instrumentation decorator on a method

We will create both successful and failing calls so that every major statistic changes.

In [38]:
class PaymentService:
    @Instrument(label="payments")
    def charge(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        return f"charged {amount}"


payments = PaymentService()

assert payments.charge(10) == "charged 10"
assert payments.charge(25) == "charged 25"

try:
    payments.charge(0)
except ValueError:
    pass

instrumented = PaymentService.__dict__["charge"]
snapshot = instrumented.snapshot()

assert snapshot["label"] == "payments"
assert snapshot["owner"] == "PaymentService"
assert snapshot["attribute_name"] == "charge"
assert snapshot["calls"] == 3
assert snapshot["exceptions"]["ValueError"] == 1

snapshot

{'label': 'payments',
 'owner': 'PaymentService',
 'attribute_name': 'charge',
 'calls': 3,
 'total_seconds': 1.5099532902240753e-05,
 'average_seconds': 5.033177634080251e-06,
 'exceptions': {'ValueError': 1}}

## Step 4 — Confirm that introspection and binding still work

Instrumentation should add behavior without destroying normal method semantics.

In [39]:
assert PaymentService.charge.__name__ == "charge"
assert "amount" in inspect.signature(PaymentService.charge).parameters
assert inspect.ismethod(payments.charge)

print(PaymentService.charge.__name__)
print(inspect.signature(PaymentService.charge))
print(payments.charge)

charge
(self, amount)
<bound method PaymentService.charge of <__main__.PaymentService object at 0x0000022AE88717F0>>


# Synthesis — A reliable mental model

A class-based decorator can be several things at once:

- a callable wrapper,
- a state container,
- a descriptor,
- a configuration target,
- an introspection object.

The subtle part is usually not implementing `__call__`.

The subtle part is deciding how the object should behave when Python retrieves it from a class.

## The central binding idea

If decorating a method replaces the original function with an arbitrary callable object, automatic method binding is lost unless the replacement participates in descriptor behavior.

A common solution is:

```python
def __get__(self, instance, owner):
    if instance is None:
        return self
    return MethodType(self, instance)
```

This restores method-like binding to the callable decorator object.

## Best-practice checklist

Before implementing a class-based decorator, ask:

1. Will it decorate only free functions, or methods too?
2. If methods are supported, what should `__get__` return?
3. Where should state live: globally, per decorated function, or per instance?
4. Should instance attributes be allowed to shadow the descriptor?
5. Must names, docs, annotations, and signatures remain visible?
6. Is configuration separate from runtime state?
7. Does stacking order matter with `classmethod` or `staticmethod`?
8. Can multiple threads update wrapper state?
9. Could references stored by the decorator keep instances alive unintentionally?
10. How will users inspect, export, or reset accumulated state?

# Extra practice drills

These are intentionally left unsolved so you can apply the patterns developed above.

### Drill A — Resettable timing statistics

Add a `reset()` method to `Timed` that safely restores all timing statistics to their initial state.

### Drill B — Per-instance timing

Store timing data separately for every instance using `WeakKeyDictionary`, and expose `stats_for(instance)`.

### Drill C — Selective exception statistics

Make `ExceptionStats` configurable so it records only a selected tuple of exception classes while still re-raising every exception.

### Drill D — Controlled class-level calls

Design a descriptor decorator that permits instance calls but raises a clear error if somebody tries to invoke the wrapper through the class directly.

### Drill E — Support two decorator syntaxes

Design a decorator class that supports both `@Decorator` and `@Decorator(option=True)`. Write down the object lifecycle before implementing it.

# Final takeaway

Decorator classes become much easier to reason about once you stop viewing them as “function decorators written with class syntax”.

They are objects participating in Python's object model.

Once such an object is stored on a class and expected to act like a method, the descriptor protocol becomes part of the design.